# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedr456/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Ranking / scoring.** The editor's question is *"which pages first?"*, not *"is this page
declining, yes or no?"*. The output is a priority score that orders a review queue; the
queue has a capacity (an editor reviews ~K pages a week), so quality is judged at the top
of the list, not across all 30,000 rows.

The one-paragraph frame (filled honestly):

> For a **FlyRank content strategist**, deciding **which pages to review first for
> refresh/expand/protect/prune**, we will build a **ranked review queue with reason codes**
> from the **starter 90-day snapshot (warehouse daily facts later)**, scoring **observed
> decline** measured by **Precision@50 under a client-holdout split**. A wrong call costs
> **a wasted editor-hour (false positive) or a silently decaying high-traffic page (false
> negative)**. A plain rule isn't enough because **the stale×visible rule scores only 17 of
> 30,000 pages and the signals interact**. We will claim only **observed / directional /
> decision-support** results.

In [1]:
# Setup + the number that makes this a RANKING problem, not plain classification.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mohamedr456/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
y = df["trend_direction"].str.lower().eq("down").astype(int)

K = 50  # weekly review capacity assumption
print(f"{len(df):,} pages, but an editor reviews ~{K}/week -> "
      f"{K/len(df):.2%} of the panel. Only the ORDER of the top matters.")
print(f"A classifier that says 'declining' for everyone is {y.mean():.1%} 'accurate' and 100% useless.")

30,000 pages, but an editor reviews ~50/week -> 0.17% of the panel. Only the ORDER of the top matters.
A classifier that says 'declining' for everyone is 54.2% 'accurate' and 100% useless.


## 2. Target or proxy

**Target (current): observed decline** — `trend_direction == "down"`, i.e. the page's
trailing-90-day trend as *measured* in the snapshot. This is an observed outcome, not a
human decision rule: no editor decided these pages were "bad", the traffic trend did.
(Contrast: FlyRank's product flags like "needs CTR fix" *would* be a defined rule — they
are deliberately excluded from the data so nobody can train on them.)

**Its known limitation (stated up front):** the label lives in the *same* 90-day window as
the features, so it says "is declining", not "will decline". That is honest for a
*review-priority* queue (review what is declining now), but it is a **proxy** for the
decision I ultimately care about. **Planned upgrade (Week 5, warehouse daily facts):** a
past→future label — features from an observation window, label = decline measured in a
*later* window — with the strict window alignment the data skill warns about.

**The label trap, enforced in code below:** `trend_direction` is bucketed from
`trend_pct`, so both are label sources and neither may ever be a feature.

In [2]:
# Where the label comes from, and the guard that keeps it out of the features.
print("Label derivation: trend_pct (x100 pct-points) -> trend_direction buckets -> binary label\n")
print(df.groupby(df["trend_direction"].str.lower())["trend_pct"]
        .agg(["count", "min", "max"]).round(1).to_string())

FEATURES = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "engagement_rate"]
LABEL_SOURCES = {"trend_direction", "trend_pct"}
assert not (set(FEATURES) & LABEL_SOURCES), "label trap sprung!"
print(f"\nLabel prevalence: {y.mean():.3f}  |  label sources {sorted(LABEL_SOURCES)} are NOT in the feature list. Guard passes.")

Label derivation: trend_pct (x100 pct-points) -> trend_direction buckets -> binary label

                 count    min      max
trend_direction                       
down             16262 -100.0    -20.0
flat                 0    NaN      NaN
new                  0    NaN      NaN
stable            5962  -20.0     20.0
up                4388   20.0  44900.0

Label prevalence: 0.542  |  label sources ['trend_direction', 'trend_pct'] are NOT in the feature list. Guard passes.


## 3. Success metric

**Precision@50 under a client-holdout split** — of the top 50 pages my queue puts in front
of an editor (from clients the model never trained on), what fraction are actually
declining?

What each bar means, computed live below:
- **Floor (random order):** a shuffled queue's top-50 lands at the base rate, ~0.54 on the
  full panel. Anything near that is worthless.
- **Reference points from the pipeline (client-holdout):** hand rule **0.240**, random
  forest **0.740** (`outputs/model_results.json`, regenerated by `scripts/run_all.py`).
- **"Good" for my project:** clearly and *stably* above both the random floor and the hand
  rule on held-out clients — with the honest caveat that heavily tied scores make P@50
  fragile (Week-1 lesson from notebook 02), so I will report it with tie handling stated.

Secondary (reported, not optimized): recall@K for the missed-decline cost, and a by-hand
review of the top 20 with reason codes — the lane's required sanity check.

In [3]:
# The metric, its random floor (simulated), and the reference bars.
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

rng = np.random.default_rng(42)
random_floor = np.mean([precision_at_k(rng.random(len(df)), y, 50) for _ in range(200)])
print(f"Random-order Precision@50 (200 shuffles): {random_floor:.3f}  <- the floor")

import json
if not os.path.exists("outputs/model_results.json"):
    subprocess.run([sys.executable, "scripts/run_all.py"], check=True, stdout=subprocess.DEVNULL)
res = json.load(open("outputs/model_results.json"))
print(f"Hand rule    (client-holdout):            {res['baseline']['baseline_precision_at_50']:.3f}")
print(f"Random forest(client-holdout):            {res['models']['random_forest']['precision_at_50']:.3f}")
print("\n'Good' = stably above the floor AND the hand rule, on held-out clients, ties handled explicitly.")

Random-order Precision@50 (200 shuffles): 0.534  <- the floor
Hand rule    (client-holdout):            0.240
Random forest(client-holdout):            0.740

'Good' = stably above the floor AND the hand rule, on held-out clients, ties handled explicitly.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (a page) for one client**, described by its
trailing-90-day search metrics. 30,000 rows, 32 clients. The grain check below proves
`content_id` is unique — no page appears twice, so no aggregation is hiding inside a row.
`client_id` exists for **grouping and splitting only** (client-holdout), never as a feature.

In [4]:
# The slice, its grain, and a look at safe columns only (pseudonymized IDs, no queries/URLs).
dup = df.groupby("content_id").size()
assert (dup == 1).all(), "grain broken: a content_id repeats"
print(f"Grain check: {df['content_id'].nunique():,} unique content_id over {len(df):,} rows -> one row = one page. OK\n")

SAFE_VIEW = ["content_id", "client_id", "content_type", "content_age_days",
             "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
print(df[SAFE_VIEW].head(5).to_string(index=False))

Grain check: 30,000 unique content_id over 30,000 rows -> one row = one page. OK

          content_id         client_id    content_type  content_age_days  days_since_last_update  impressions_90d  avg_position  ctr  word_count
content_304f48230142 client_f369cb89fc keyword article               187                      20             3803          10.6 0.76      3221.0
content_a1fb4e703a9e client_4e07408562 keyword article               445                      25            15320          20.3 0.05      2481.0
content_9aa793d4d895 client_7f2253d7e2 keyword article               141                      20            12581          36.5 0.09      3515.0
content_331d6c4de07b client_19581e27de keyword article               463                      22            11751           6.2 0.49         NaN
content_d99b7a2d90ca client_3fdba35f04 keyword article               263                      14            19140          44.0 0.13      2803.0


## 5. Why ML beats a fixed rule here

Because the signals **interact**, and the hand rule's one true cell is **too small to feed
a queue**. The table below (computed live) shows it:

- The rule's target cell — stale>180d AND high-visibility — has the highest decline rate
  (**0.941**)… across just **17 pages**. The rule is *right* and *starving*: it fills a
  top-50 queue from 17 scored pages plus ties.
- Meanwhile decline runs **0.58–0.65** in the huge cells the rule ignores completely
  (fresh/aging × mid/high visibility — over 20,000 pages), so almost all recoverable
  traffic lives where the rule never looks.
- And staleness is **not monotonic** within a visibility band (low-visibility: 0.367 →
  0.526 → 0.403), so "raise the staleness threshold" cannot fix it.

Add position tier, content type (my Week-1 discovery: CTR differs by type *at the same
tier*), engagement and age, and the pattern is real but too tangled for if-statements —
the exact condition under which a small readable model earns its keep. A depth-limited
tree or logistic model can *find* the interaction boundaries and still be printed and read
(Week-1 notebook 02 lesson), which keeps reason codes possible.

In [5]:
# The interaction the hand rule can't see: decline rate by staleness x visibility.
stale_band = pd.cut(df["days_since_last_update"], [-1, 90, 180, 10_000],
                    labels=["fresh<=90d", "aging 91-180d", "stale>180d"])
vis_band = pd.cut(df["impressions_90d"], [-1, 99, 499, 10_000_000],
                  labels=["low<100", "mid 100-499", "high>=500"])

table = df.assign(stale=stale_band, vis=vis_band).pivot_table(
    index="stale", columns="vis", values="trend_direction",
    aggfunc=lambda s: s.str.lower().eq("down").mean(), observed=True)
print("Decline rate by staleness x visibility (row = staleness, col = visibility):\n")
print(table.round(3).to_string())

cell_n = df.assign(stale=stale_band, vis=vis_band).pivot_table(
    index="stale", columns="vis", values="content_id", aggfunc="count", observed=True)
print("\nRow counts per cell:\n"); print(cell_n.to_string())
print("\nObserved: the rule's one hot cell (0.94 decline) holds 17 pages; the cells it "
      "ignores hold 20,000+ pages at 0.58-0.65 decline; staleness is non-monotonic at low "
      "visibility -> no threshold tweak fixes this, a learned scorer can.")

Decline rate by staleness x visibility (row = staleness, col = visibility):

vis            low<100  mid 100-499  high>=500
stale                                         
fresh<=90d       0.367        0.585      0.582
aging 91-180d    0.526        0.652      0.616
stale>180d       0.403        0.556      0.941

Row counts per cell:

vis            low<100  mid 100-499  high>=500
stale                                         
fresh<=90d        6768         3736      10151
aging 91-180d     1087         1526       6558
stale>180d         139           18         17

Observed: the rule's one hot cell (0.94 decline) holds 17 pages; the cells it ignores hold 20,000+ pages at 0.58-0.65 decline; staleness is non-monotonic at low visibility -> no threshold tweak fixes this, a learned scorer can.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.